# YOLOv11 Inference and ONNX Export

This notebook prepares the best YOLOv8 detector for deployment. It downloads the selected PyTorch checkpoint, exports the model to ONNX with preprocessing-compatible input dimensions and embedded non-maximum suppression, and validates the exported graph against the TACO validation split.

The exported ONNX model can be executed with ONNX Runtime in an application container that does not include PyTorch or Ultralytics. The accompanying `class_names.json` file preserves the mapping from numeric prediction IDs to the reduced TACO class names.

## Install Packages

Install the libraries required to load the training checkpoint, export it to ONNX, and evaluate the exported model. Ultralytics performs the conversion and may install ONNX-specific export dependencies when they are not already available in the runtime. These training-side dependencies are only needed during export; the final deployment container can use ONNX Runtime without PyTorch.

In [1]:
!pip install torchmetrics pycocotools gdown tqdm tensorboard albumentations ultralytics

## Import packages

In [8]:
import os
from pathlib import Path
import json
import shutil
from ultralytics import YOLO

## Setup environment

In [9]:
def is_google_colab() -> bool:
    """Check whether the notebook is running in Google Colab.
    
    Returns:
        bool: ``True`` in a Colab runtime; otherwise ``False``.
    """
    try:
        import google.colab
        return True
    except ImportError:
        return False


IN_COLAB = is_google_colab()

if IN_COLAB:
    ROOT_PATH = "/content"
    DATAFRAME_PATH = "/content/drive/MyDrive/taco_trash"
    DEFAULT_RUNS_PATH = "/content/drive/MyDrive/taco_trash"
else:
    ROOT_PATH = "/home/ubuntu"
    DATAFRAME_PATH = os.path.join(ROOT_PATH, "taco_trash")
    DEFAULT_RUNS_PATH = os.path.join(ROOT_PATH, "taco_trash")

TACO_CLASSIFICATION_PATH = os.path.join(ROOT_PATH, "taco")

In [13]:
if is_google_colab():
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
else:
    !curl https://rclone.org/install.sh | sudo bash

Mounted at /content/drive


## Download the Best Model

Download the `best.pt` checkpoint selected by validation performance during training. This checkpoint corresponds to the best epoch rather than the final epoch, which avoids deploying the lower-performing weights produced after the validation metric had plateaued.

The downloaded file is stored as `./best.pt` and becomes the source artifact for ONNX export. In a production workflow, the file should be versioned together with its taxonomy, image size, and validation metrics.

In [2]:
# !https://drive.google.com/file/d/10jKFgBIAnJa_YlwFeQO_rnwPUHEaJjOH/view?usp=sharing
!gdown 10jKFgBIAnJa_YlwFeQO_rnwPUHEaJjOH

Downloading...
From (original): https://drive.google.com/uc?id=10jKFgBIAnJa_YlwFeQO_rnwPUHEaJjOH
From (redirected): https://drive.google.com/uc?id=10jKFgBIAnJa_YlwFeQO_rnwPUHEaJjOH&confirm=t&uuid=d2f6c324-c1fb-44e3-be93-387cd4f0d6c1
To: /content/best.pt
100% 51.2M/51.2M [00:00<00:00, 162MB/s]


## Export to ONNX

Load the PyTorch checkpoint and convert it into a fixed-shape ONNX graph accepting one `1024 × 1024` image at a time. Graph simplification improves runtime compatibility, while embedded non-maximum suppression returns final detections instead of the model's raw candidate boxes. The export also writes `class_names.json`, which must be deployed with the ONNX file.

The configured confidence threshold (`0.25`), IoU threshold (`0.7`), and maximum detection count (`300`) become part of the exported postprocessing behavior when NMS is embedded. Predictions below the export confidence threshold cannot be recovered later by passing a lower threshold to ONNX Runtime. Export with a lower confidence threshold or without embedded NMS if the serving application must tune this value dynamically.

In [14]:
import json
from ultralytics import YOLO

checkpoint = "./best.pt"
drive_export_dir = Path("/content/drive/MyDrive/taco_yolo_runs/yolo11l_top5_1280")
drive_export_dir.mkdir(parents=True, exist_ok=True)

model = YOLO(checkpoint)

onnx_path = model.export(
    format="onnx",
    imgsz=1280,
    opset=12,
    simplify=True,
    dynamic=False,
    nms=True,
    conf=0.001,
    iou=0.7,
    max_det=300
)
onnx_path = Path(onnx_path)

class_names_path = Path("class_names.json")
with open("class_names.json", "w") as file:
    json.dump(model.names, file)

print(onnx_path)
shutil.copy2(onnx_path, drive_export_dir / onnx_path.name)
shutil.copy2(class_names_path, drive_export_dir / class_names_path.name)

print("ONNX copied to:", drive_export_dir / onnx_path.name)
print("Class names copied to:", drive_export_dir / class_names_path.name)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11l summary (fused): 191 layers, 25,280,854 parameters, 0 gradients, 86.6 GFLOPs

PyTorch: starting from 'best.pt' with input shape (1, 3, 1280, 1280) BCHW and output shape(s) (1, 300, 6) (48.9 MB)

ONNX: starting export with onnx 1.22.0 opset 12...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 6.0s, saved as 'best.onnx' (97.3 MB)

Export complete (9.0s)
Results saved to /content/best.onnx
Predict:         yolo predict task=detect model=best.onnx imgsz=1280 
Validate:        yolo val task=detect model=best.onnx imgsz=1280 data=/content/taco_yolo_top5/taco.yaml  
Visualize:       https://netron.app
best.onnx
ONNX copied to: /content/drive/MyDrive/taco_yolo_runs/yolo11l_top5_1280/best.onnx
Class names copied to: /content/drive/MyDrive/taco_yolo_runs/yolo11l_top5_1280/class_names.json


## Validate the Exported Model

Reload the ONNX artifact through Ultralytics and run it on the same validation split used for the final PyTorch checkpoint. This checks that conversion succeeded, all operators are supported, and the exported model still produces reasonable bounding-box metrics before it is copied into the deployment image.

Use the same image size, batch size, IoU threshold, and maximum detection count as the reference validation whenever metric parity is required. Because this graph was exported with embedded NMS at confidence `0.25`, setting `conf=0.001` in this validation call cannot restore candidates already removed inside the graph. A strict PyTorch-versus-ONNX comparison therefore requires exporting with `conf=0.001` or exporting without NMS and performing postprocessing outside the graph.

In [16]:
YOLO_TOP5_DIR = Path("/content/taco_yolo_top5")

SELECTED_CLASSES = [
    "Can",
    "Plastic bottle",
    "Plastic bottle cap",
    "Cup",
    "Plastic film",
]

MAP_TOP5 = {
    class_name: class_name
    for class_name in SELECTED_CLASSES
}

In [17]:
import json
import os
from pathlib import Path
import shutil

import torch


def convert_taco_to_yolo(
    taco_dir,
    output_dir,
    category_mapping,
    train_fraction=0.8,
    seed=42,
    image_mode="copy",
    drop_other=False,
    drop_empty_images=False,
):
    """Convert COCO-style TACO annotations to YOLO detection format.

    Args:
        taco_dir (str | Path): Directory containing `annotations.json`
            and image folders such as `batch_1`, `batch_2`, etc.
        output_dir (str | Path): Destination directory for the YOLO dataset.
        category_mapping (dict[str, str]): Mapping from original TACO category
            names to the target/reduced class names.
        train_fraction (float): Fraction of images used for training.
        seed (int): Seed for deterministic image-level split.
        image_mode (str): How images are materialized. One of `copy`,
            `symlink`, or `hardlink`.
        drop_other (bool): If True, annotations mapped to `Other` are removed.
        drop_empty_images (bool): If True, images with no remaining annotations
            after filtering are skipped.

    Returns:
        Path: Path to generated `taco.yaml`.
    """
    taco_dir = Path(taco_dir)
    output_dir = Path(output_dir)

    with (taco_dir / "annotations.json").open() as file:
        coco = json.load(file)

    images = {
        int(image["id"]): image
        for image in coco["images"]
    }

    categories = {
        int(category["id"]): category["name"]
        for category in coco["categories"]
    }

    annotations_by_image = {}
    for annotation in coco["annotations"]:
        image_id = int(annotation["image_id"])
        annotations_by_image.setdefault(image_id, []).append(annotation)

    mapped_names = {
        category_mapping.get(name, "Other")
        for name in categories.values()
    }

    if drop_other:
        mapped_names.discard("Other")

    class_names = sorted(mapped_names)
    name_to_id = {
        name: class_id
        for class_id, name in enumerate(class_names)
    }

    category_id_to_yolo_id = {}

    for category_id, category_name in categories.items():
        mapped_name = category_mapping.get(category_name, "Other")

        if drop_other and mapped_name == "Other":
            category_id_to_yolo_id[category_id] = None
        else:
            category_id_to_yolo_id[category_id] = name_to_id.get(mapped_name)

    image_ids = sorted(images)

    generator = torch.Generator().manual_seed(seed)
    permutation = torch.randperm(
        len(image_ids),
        generator=generator,
    ).tolist()

    train_size = int(train_fraction * len(image_ids))

    train_ids = {
        image_ids[index]
        for index in permutation[:train_size]
    }

    val_ids = {
        image_ids[index]
        for index in permutation[train_size:]
    }

    def materialize_image(source, destination):
        destination.parent.mkdir(parents=True, exist_ok=True)

        if destination.exists() or destination.is_symlink():
            destination.unlink()

        if image_mode == "copy":
            shutil.copy2(source, destination)
        elif image_mode == "hardlink":
            os.link(source, destination)
        elif image_mode == "symlink":
            destination.symlink_to(source.resolve())
        else:
            raise ValueError(f"Unknown image_mode: {image_mode}")

    written_annotations = 0
    skipped_boxes = 0
    written_images = 0
    skipped_empty_images = 0

    for image_id, image in images.items():
        split = "train" if image_id in train_ids else "val"

        relative_path = Path(image["file_name"])
        source_path = taco_dir / relative_path

        image_path = output_dir / "images" / split / relative_path
        label_path = (
            output_dir
            / "labels"
            / split
            / relative_path.with_suffix(".txt")
        )

        image_width = int(image["width"])
        image_height = int(image["height"])

        label_lines = []

        for annotation in annotations_by_image.get(image_id, []):
            class_id = category_id_to_yolo_id[
                int(annotation["category_id"])
            ]

            if class_id is None:
                continue

            x, y, width, height = map(float, annotation["bbox"])

            x1 = min(max(x, 0.0), image_width)
            y1 = min(max(y, 0.0), image_height)
            x2 = min(max(x + width, 0.0), image_width)
            y2 = min(max(y + height, 0.0), image_height)

            width = x2 - x1
            height = y2 - y1

            if width <= 0 or height <= 0:
                skipped_boxes += 1
                continue

            x_center = ((x1 + x2) / 2) / image_width
            y_center = ((y1 + y2) / 2) / image_height
            width_normalized = width / image_width
            height_normalized = height / image_height

            label_lines.append(
                f"{class_id} "
                f"{x_center:.8f} "
                f"{y_center:.8f} "
                f"{width_normalized:.8f} "
                f"{height_normalized:.8f}"
            )

            written_annotations += 1

        if drop_empty_images and len(label_lines) == 0:
            skipped_empty_images += 1
            continue

        materialize_image(source_path, image_path)

        label_path.parent.mkdir(parents=True, exist_ok=True)
        label_path.write_text(
            "\n".join(label_lines) + ("\n" if label_lines else "")
        )

        written_images += 1

    output_dir.mkdir(parents=True, exist_ok=True)

    yaml_lines = [
        f'path: "{output_dir.resolve()}"',
        "train: images/train",
        "val: images/val",
        "names:",
    ]

    for class_id, class_name in enumerate(class_names):
        yaml_lines.append(
            f"  {class_id}: {json.dumps(class_name)}"
        )

    yaml_path = output_dir / "taco.yaml"
    yaml_path.write_text("\n".join(yaml_lines) + "\n")

    manifest = {
        "seed": seed,
        "train_fraction": train_fraction,
        "train_image_ids": sorted(train_ids),
        "val_image_ids": sorted(val_ids),
        "classes": class_names,
        "written_images": written_images,
        "written_annotations": written_annotations,
        "skipped_boxes": skipped_boxes,
        "skipped_empty_images": skipped_empty_images,
        "drop_other": drop_other,
        "drop_empty_images": drop_empty_images,
    }

    (output_dir / "split.json").write_text(
        json.dumps(manifest, indent=2)
    )

    print(f"Classes: {class_names}")
    print(f"Train images before filtering: {len(train_ids)}")
    print(f"Val images before filtering: {len(val_ids)}")
    print(f"Written images: {written_images}")
    print(f"Written annotations: {written_annotations}")
    print(f"Skipped invalid boxes: {skipped_boxes}")
    print(f"Skipped empty images: {skipped_empty_images}")
    print(f"YAML: {yaml_path}")

    return yaml_path

In [18]:
from pathlib import Path

TACO_DIR = Path(
    "/content/taco"
)

YOLO_DIR = Path("/content/taco_yolo")

YOLO_TOP5_DIR = Path("/content/taco_yolo_top5")

SELECTED_CLASSES = [
    "Can",
    "Plastic bottle",
    "Plastic bottle cap",
    "Cup",
    "Plastic film",
]

MAP_TOP5 = {
    class_name: class_name
    for class_name in SELECTED_CLASSES
}

yaml_path_top5 = convert_taco_to_yolo(
    taco_dir=TACO_DIR,
    output_dir=YOLO_TOP5_DIR,
    category_mapping=MAP_TOP5,
    train_fraction=0.8,
    seed=42,
    image_mode="copy",
    drop_other=True,
    drop_empty_images=True,
)

print(Path(yaml_path_top5).read_text())

Classes: ['Plastic bottle cap', 'Plastic film']
Train images before filtering: 1200
Val images before filtering: 300
Written images: 474
Written annotations: 660
Skipped invalid boxes: 0
Skipped empty images: 1026
YAML: /content/taco_yolo_top5/taco.yaml
path: "/content/taco_yolo_top5"
train: images/train
val: images/val
names:
  0: "Plastic bottle cap"
  1: "Plastic film"



In [19]:
onnx_model = YOLO("/content/drive/MyDrive/taco_yolo_runs/yolo11l_top5_1280/best.onnx")

onnx_metrics = onnx_model.val(
    data=str(yaml_path_top5),
    imgsz=1280,
    batch=4,
    conf=0.001,
    iou=0.7,
    max_det=300,
    device=0,  # use "cpu" if CUDA/ONNXRuntime GPU is unavailable
    plots=True,
)

print("ONNX Precision:", onnx_metrics.box.mp)
print("ONNX Recall:", onnx_metrics.box.mr)
print("ONNX mAP@0.5:", onnx_metrics.box.map50)
print("ONNX mAP@0.5:0.95:", onnx_metrics.box.map)

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Loading /content/drive/MyDrive/taco_yolo_runs/yolo11l_top5_1280/best.onnx for ONNX Runtime inference...
WARNING ⚠️ CUDA requested but CUDAExecutionProvider not available. Using CPU...
Using ONNX Runtime 1.27.0 with CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 1280, 1280)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3797.7±313.4 MB/s, size: 2294.3 KB)
val: Scanning /content/taco_yolo_top5/labels/val/batch_1... 89 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 89/89 1.1Kit/s 0.1s
val: New cache created: /content/taco_yolo_top5/labels/val/batch_1.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 89/89 1.2s/it 1:481.3sss
   